In [2]:

import pandas as pd
import numpy as np

# loading data
df = pd.read_csv("data/Raw_Data/train.csv")
df.drop(columns=["Unnamed: 0"], inplace=True)

print("=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)
print(f"Shape : {df.shape}")
print(df.head(3).to_string())



ORIGINAL DATASET
Shape : (5847, 13)
                               Name Location  Year  Kilometers_Driven Fuel_Type Transmission Owner_Type     Mileage   Engine      Power  Seats  New_Price  Price
0  Hyundai Creta 1.6 CRDi SX Option     Pune  2015              41000    Diesel       Manual      First  19.67 kmpl  1582 CC  126.2 bhp    5.0        NaN   12.5
1                      Honda Jazz V  Chennai  2011              46000    Petrol       Manual      First    13 km/kg  1199 CC   88.7 bhp    5.0  8.61 Lakh    4.5
2                 Maruti Ertiga VDI  Chennai  2012              87000    Diesel       Manual      First  20.77 kmpl  1248 CC  88.76 bhp    7.0        NaN    6.0


In [ ]:

# (a) Handle Missing Values

print("\n" + "=" * 70)
print("(a) MISSING VALUES – BEFORE")
print("=" * 70)
print(df.isnull().sum())



(a) MISSING VALUES – BEFORE
Name                    0
Location                0
Year                    0
Kilometers_Driven       0
Fuel_Type               0
Transmission            0
Owner_Type              0
Mileage                 2
Engine                 36
Power                  36
Seats                  38
New_Price            5032
Price                   0
dtype: int64


In [ ]:

#Mileage (2 missing)

mileage_numeric = df["Mileage"].str.extract(r"([\d.]+)")[0].astype(float)
df["Mileage"] = df["Mileage"].fillna(str(mileage_numeric.median()) + " kmpl")

#Engine (36 missing)
engine_numeric = df["Engine"].str.extract(r"([\d.]+)")[0].astype(float)
engine_median= engine_numeric.median()
df["Engine"] = df["Engine"].fillna(f"{engine_median} CC")

#Power (36 missing)
df["Power"] = df["Power"].replace("null bhp", np.nan)
power_numeric = df["Power"].str.extract(r"([\d.]+)")[0].astype(float)
power_median = power_numeric.median()
df["Power"] = df["Power"].fillna(f"{power_median} bhp")

#Seats (38 missing)
seats_mode = df["Seats"].mode()[0]
df["Seats"] = df["Seats"].fillna(seats_mode)

#New_Price (5032 missing – 86 %)
df.drop(columns=["New_Price"], inplace=True)

print("\nMissing values – AFTER cleaning")
print(df.isnull().sum())



Missing values – AFTER cleaning
Name                 0
Location             0
Year                 0
Kilometers_Driven    0
Fuel_Type            0
Transmission         0
Owner_Type           0
Mileage              0
Engine               0
Power                0
Seats                0
Price                0
dtype: int64


In [3]:


# (b) Removing Units – Keeping Numerical Values Only
print("\n" + "=" * 70)
print("(b) REMOVING UNITS")
print("=" * 70)

# Mileage: "19.67 kmpl" or "13 km/kg" -  float
df["Mileage"] = df["Mileage"].str.extract(r"([\d.]+)").astype(float)

# Engine:  "1582 CC" - float
df["Engine"] = df["Engine"].str.extract(r"([\d.]+)").astype(float)

# Power:   "126.2 bhp" - float
df["Power"] = df["Power"].str.extract(r"([\d.]+)").astype(float)

# Seats is already numeric after fillna; use nullable Int64 for safety
df["Seats"] = df["Seats"].astype("Int64")

print("Dtypes after unit removal:")
print(df[["Mileage", "Engine", "Power", "Seats"]].dtypes)
print(df[["Mileage", "Engine", "Power", "Seats"]].describe())




(b) REMOVING UNITS
Dtypes after unit removal:
Mileage    float64
Engine     float64
Power      float64
Seats        Int64
dtype: object
           Mileage       Engine        Power     Seats
count  5845.000000  5811.000000  5811.000000    5809.0
mean     18.158496  1631.552573   113.803144  5.286452
std       4.358246   601.972587    53.896719  0.806668
min       0.000000    72.000000    34.200000       2.0
25%      15.260000  1198.000000    78.000000       5.0
50%      18.190000  1497.000000    98.600000       5.0
75%      21.100000  1991.000000   139.010000       5.0
max      28.400000  5998.000000   560.000000      10.0


In [4]:

# (c) One-Hot Encode Categorical Variables: Fuel_Type & Transmission
print("\n"+"=" * 70)
print("(c) ONE-HOT ENCODING  Fuel_Type & Transmission")
print("=" * 70)
# pd.get_dummies creates binary columns and drops the originals when
# drop_first=False (we keep all dummies for transparency).
df = pd.get_dummies(df, columns=["Fuel_Type", "Transmission"], dtype=int)

ohe_cols = [c for c in df.columns if c.startswith("Fuel_Type") or c.startswith("Transmission")]
print("New OHE columns:", ohe_cols)
print(df[ohe_cols].head(5).to_string())






(c) ONE-HOT ENCODING  Fuel_Type & Transmission
New OHE columns: ['Fuel_Type_Diesel', 'Fuel_Type_Electric', 'Fuel_Type_Petrol', 'Transmission_Automatic', 'Transmission_Manual']
   Fuel_Type_Diesel  Fuel_Type_Electric  Fuel_Type_Petrol  Transmission_Automatic  Transmission_Manual
0                 1                   0                 0                       0                    1
1                 0                   0                 1                       0                    1
2                 1                   0                 0                       0                    1
3                 1                   0                 0                       1                    0
4                 1                   0                 0                       0                    1


In [5]:

# (d) Feature Engineering – Car Age
print("\n" + "=" * 70)
print("(d) FEATURE ENGINEERING - Car_Age")
print("=" * 70)

CURRENT_YEAR = 2026
df["Car_Age"] = CURRENT_YEAR - df["Year"]

print("Sample Car_Age values:")
print(df[["Name", "Year", "Car_Age"]].head(5).to_string())
print(f"\nCar_Age – min: {df['Car_Age'].min()},  max: {df['Car_Age'].max()},  mean: {df['Car_Age'].mean():.1f}")




(d) FEATURE ENGINEERING - Car_Age
Sample Car_Age values:
                               Name  Year  Car_Age
0  Hyundai Creta 1.6 CRDi SX Option  2015       11
1                      Honda Jazz V  2011       15
2                 Maruti Ertiga VDI  2012       14
3   Audi A4 New 2.0 TDI Multitronic  2013       13
4            Nissan Micra Diesel XV  2013       13

Car_Age – min: 7,  max: 28,  mean: 12.6


In [6]:

# (E)

print("\n" + "=" * 70)
print("(e) DATA MANIPULATION OPERATIONS")
print("=" * 70)

select_cols = ["Name", "Location", "Year", "Car_Age",
               "Kilometers_Driven", "Mileage", "Engine",
               "Power", "Seats", "Price"]
df_sel = df[select_cols].copy()
print("\n[SELECT] – Chosen columns:", select_cols)
print(df_sel.head(3).to_string())



(e) DATA MANIPULATION OPERATIONS

[SELECT] – Chosen columns: ['Name', 'Location', 'Year', 'Car_Age', 'Kilometers_Driven', 'Mileage', 'Engine', 'Power', 'Seats', 'Price']
                               Name Location  Year  Car_Age  Kilometers_Driven  Mileage  Engine   Power  Seats  Price
0  Hyundai Creta 1.6 CRDi SX Option     Pune  2015       11              41000    19.67  1582.0  126.20      5   12.5
1                      Honda Jazz V  Chennai  2011       15              46000    13.00  1199.0   88.70      5    4.5
2                 Maruti Ertiga VDI  Chennai  2012       14              87000    20.77  1248.0   88.76      7    6.0


In [7]:


# Keep only cars with price ≤ 20 lakh and with 5 seats (most common segment).
df_filt = df_sel[(df_sel["Price"] <= 20) & (df_sel["Seats"] == 5)].copy()
print(f"\nFILTER – Price ≤ 20 lakh & Seats == 5   {len(df_filt)} rows  (from {len(df_sel)})")
print(df_filt.head(3).to_string())



FILTER – Price ≤ 20 lakh & Seats == 5   4371 rows  (from 5847)
                               Name    Location  Year  Car_Age  Kilometers_Driven  Mileage  Engine  Power  Seats  Price
0  Hyundai Creta 1.6 CRDi SX Option        Pune  2015       11              41000    19.67  1582.0  126.2      5  12.50
1                      Honda Jazz V     Chennai  2011       15              46000    13.00  1199.0   88.7      5   4.50
3   Audi A4 New 2.0 TDI Multitronic  Coimbatore  2013       13              40670    15.20  1968.0  140.8      5  17.74


In [15]:


# Make column names more descriptive / Pythonic.
df_ren = df_filt.rename(columns={
    "Kilometers_Driven" : "KM_Driven",
    "Price"             : "Price_Lakh"
})
print("\nRENAME – Renamed columns: Kilometers_Driven - KM_Driven, Price - Price_Lakh")
print(df_ren.head(3).to_string())



RENAME – Renamed columns: Kilometers_Driven - KM_Driven, Price - Price_Lakh
                               Name    Location  Year  Car_Age  KM_Driven  Mileage  Engine  Power  Seats  Price_Lakh
0  Hyundai Creta 1.6 CRDi SX Option        Pune  2015       11      41000    19.67  1582.0  126.2      5       12.50
1                      Honda Jazz V     Chennai  2011       15      46000    13.00  1199.0   88.7      5        4.50
3   Audi A4 New 2.0 TDI Multitronic  Coimbatore  2013       13      40670    15.20  1968.0  140.8      5       17.74


In [14]:


# Add a new derived column: price per kilometre (value retention proxy).
df_mut = df_ren.copy()
df_mut["Price_per_KM"] = (df_mut["Price_Lakh"] / df_mut["KM_Driven"]).round(6)
print("\nMUTATE – Added 'Price_per_KM' column (Price_Lakh / KM_Driven)")
print(df_mut[["Name", "Price_Lakh", "KM_Driven", "Price_per_KM"]].head(5).to_string())



MUTATE – Added 'Price_per_KM' column (Price_Lakh / KM_Driven)
                                  Name  Price_Lakh  KM_Driven  Price_per_KM
0     Hyundai Creta 1.6 CRDi SX Option       12.50      41000      0.000305
1                         Honda Jazz V        4.50      46000      0.000098
3      Audi A4 New 2.0 TDI Multitronic       17.74      40670      0.000436
4               Nissan Micra Diesel XV        3.50      86999      0.000040
6  Volkswagen Vento Diesel Comfortline        5.20      64430      0.000081


In [13]:


# Sort descending by Price_Lakh, then ascending by Car_Age.
df_arr = df_mut.sort_values(["Price_Lakh", "Car_Age"], ascending=[False, True])
print("\nARRANGE – Sorted by Price_Lakh (desc), Car_Age (asc)")
print(df_arr[["Name", "Price_Lakh", "Car_Age"]].head(5).to_string())



ARRANGE – Sorted by Price_Lakh (desc), Car_Age (asc)
                                     Name  Price_Lakh  Car_Age
580            Toyota Corolla Altis VL AT        20.0        8
907          Jeep Compass 2.0 Limited 4X4        20.0        9
3435  Jeep Compass 2.0 Limited Option 4X4        20.0        9
360       Mercedes-Benz GLA Class 200 CDI        20.0       11
3134                 Volvo S60 D4 KINETIC        20.0       11


In [11]:

# Group by Location; compute count, mean price, median KM driven, mean mileage.
summary = (
    df_mut
    .groupby("Location")
    .agg(
        Car_Count = ("Name", "count"),
        Avg_Price  = ("Price_Lakh",  "mean"),
        Median_KM  = ("KM_Driven", "median"),
        Avg_Mileage  = ("Mileage",  "mean")
    )
    .round(2)
    .sort_values("Avg_Price", ascending=False)
    .reset_index()
)
print("\n[SUMMARISE with GROUP BY] – Grouped by Location")
print(summary.to_string(index=False))



[SUMMARISE with GROUP BY] – Grouped by Location
  Location  Car_Count  Avg_Price  Median_KM  Avg_Mileage
Coimbatore        399       7.51    44487.0        19.37
 Bangalore        220       6.91    58000.0        18.33
     Kochi        475       6.81    43828.0        19.64
 Ahmedabad        170       6.48    55052.5        19.88
    Mumbai        588       6.34    41000.0        17.87
 Hyderabad        518       5.79    68000.0        20.00
     Delhi        380       5.64    60000.0        19.21
   Chennai        370       5.07    68000.0        19.21
      Pune        468       4.98    64659.5        18.64
    Jaipur        326       4.61    66061.5        20.17
   Kolkata        457       4.44    38002.0        19.82


In [12]:


# Save cleaned dataset
output_path = "data/Cleaned_Data/used_cars_cleaned.csv"
df.to_csv(output_path, index=False)
print("\n" + "=" * 70)
print(f"Cleaned dataset saved to  {output_path}")
print(f"Final shape: {df.shape}")
print("=" * 70)


Cleaned dataset saved to  data/Cleaned_Data/used_cars_cleaned.csv
Final shape: (5847, 17)
